# S4. Configuring an algorithm, in Python

The companion of VAMOS-Studio's tutorial *S4. Exploring a parameter space*. In VAMOS an algorithm's configuration is an object such as `NSGAIIConfig`, built from its default or with a builder; the studio's parameter spaces describe the same choices as a grammar.

In [ ]:
from importlib.resources import files

import numpy as np
import plotly.graph_objects as go
import vamos
from vamos.algorithms import NSGAIIConfig
from vamos.foundation.quality_indicators import get_indicator

print("VAMOS", vamos.__version__)


def reference_front(name: str) -> np.ndarray:
    """A reference front copied from jMetal, as VAMOS-Studio ships it (e.g. "ZDT1.csv")."""
    path = files("vamos_studio") / "resources" / "referenceFronts" / name
    return np.loadtxt(path, delimiter=",", ndmin=2)


def indicators(front: np.ndarray, reference: np.ndarray) -> dict[str, float]:
    """HV and IGD+ as VAMOS-Studio computes them: objectives normalized by the reference front."""
    lower = reference.min(axis=0)
    span = reference.max(axis=0) - lower
    span[span == 0] = 1.0
    normalized, normalized_reference = (front - lower) / span, (reference - lower) / span
    hv = get_indicator("hv", reference_point=np.ones(front.shape[1]))
    igd_plus = get_indicator("igd_plus", reference_front=normalized_reference)
    return {
        "HV": round(float(hv.compute(np.clip(normalized, 0.0, 1.0)).value), 4),
        "IGD+": round(float(igd_plus.compute(normalized).value), 4),
    }


def plot(front: np.ndarray, reference: np.ndarray, title: str) -> go.Figure:
    """A two-objective front in blue over its reference front in grey."""
    figure = go.Figure()
    figure.add_scatter(
        x=reference[:, 0],
        y=reference[:, 1],
        mode="markers",
        name="Reference front",
        marker={"size": 3, "color": "#b7bcc5"},
    )
    figure.add_scatter(
        x=front[:, 0],
        y=front[:, 1],
        mode="markers",
        name="Front",
        marker={"size": 7, "color": "#1769aa"},
    )
    figure.update_layout(title=title, xaxis_title="f1", yaxis_title="f2")
    return figure

## The default configuration

Every field of the configuration is one of the choices the parameter space describes:

In [ ]:
from vamos.algorithms import available_crossover_methods

default = NSGAIIConfig.default(pop_size=100, n_var=30)
for field in ("pop_size", "crossover", "mutation", "selection", "offspring_size"):
    print(f"{field:15s}", getattr(default, field))
print("crossovers for real-coded problems:", available_crossover_methods("real"))

## A configuration with a builder

The builder sets one choice at a time, and each operator takes its own parameters, as each value of the grammar brings its own:

In [ ]:
config = (
    NSGAIIConfig.builder()
    .pop_size(50)
    .crossover("blx_alpha", prob=0.9, alpha=0.5)
    .mutation("pm", prob=1 / 30, eta=20.0)
    .selection("tournament", size=2)
    .build()
)
zdt1 = reference_front("ZDT1.csv")
for name, algorithm_config in (("default", default), ("built", config)):
    result = vamos.optimize(
        "zdt1",
        algorithm="nsgaii",
        algorithm_config=algorithm_config,
        max_evaluations=25000,
        seed=1,
    )
    print(name, indicators(np.asarray(result.F), zdt1))
# default: HV 0.6593, IGD+ 0.0038; built: HV 0.6495, IGD+ 0.0091 (S2's other configuration)

## Next

The other algorithms have their own configuration classes (`MOEADConfig`, `SMSEMOAConfig`, `NSGAIIIConfig`, `RVEAConfig`…) in `vamos.algorithms`, with `default()` and `builder()` as NSGA-II's.